# Lesson 4: Pre-key-on readiness

After the [health survey](03_health_check.ipynb), check warm-up progress and
readiness before any physical key-on action. This notebook sends queries only.
It does not turn the key, enable emission, open the shutter or change a setting.

Run cells in order with one serial owner and the instrument prompt enabled.
Every exchange waits for the full reply and following prompt. Replies remain raw
bytes: no automatic encoding, decoding or readiness decision.
If communication fails, inspect the evidence and disconnect; do not retry.

## 1. Connection settings

Use the port identified in lesson 0 and confirm model and baud on the instrument.
No fault-clear text is required to collect raw replies. Interpret their meaning
with the operator and the [manual](../verdi.manual_v5.pdf).

In [1]:
PORT = "COM5"
MODEL = "V2"  # V2, V5 or V6: match the actual laser
BAUDRATE = 19200  # Match the front panel
TIMEOUT_S = 2.0

## 2. Create a read-only controller

Construction opens nothing. Writes remain disabled.

In [2]:
from coherent_verdi import VerdiController

if PORT == "<PORT>":
    raise ValueError("Edit the connection settings in section 1 first.")
laser = VerdiController(
    PORT,
    model=MODEL,
    baudrate=BAUDRATE,
    timeout_s=TIMEOUT_S,
    prompt_enabled=True,
)

## 3. Open the serial port

Run this cell by itself and inspect the result before starting serial communication.
`connect()` opens the local serial port and returns `True` on success. Failures
raise an exception. It sends no instrument instructions. A successful open does
not establish that the laser is responding or ready.

Close any other serial owner first. If opening fails, stop here.

`override=True` closes an older controller connection to the same port in this
Python process before opening the new one. It cannot take ownership from another
kernel/application or a raw `ser` handle; release those explicitly. A failed
controller still requires a fresh controller object. Reopening does not establish
instrument synchronization or repair corrupted replies.

In [3]:
connected = laser.connect(override=True)
print("Serial port open:", connected)

Serial port open: True


## 4. Start serial communication

Only after inspecting the connection result, run the following cells separately.
Buffer maintenance and instrument queries belong here, outside the connection
section. An explicit purge discards host queues; it does not reset the device's
parser. No delay or purge is claimed to prove synchronization. Each query waits
for its complete reply and prompt before another can be sent.

In [4]:
laser.purge_buffers()  # Explicit host TX/RX cleanup before the first request.
version_reply = laser.read(b"?SV")
print(repr(version_reply))

b'6.05'


## 5. Before turning the key ON

**Keep the key in STANDBY until the front panel changes from “System warming up”
to “Standby.”** Premature key-on can cause an **LBO Not Locked at Set Temp** fault.
Allow about 30 minutes after a cold start, but verify actual readiness rather than
relying only on elapsed time. Turning the key ON can enable emission.
[Manual, pp. 4-2–4-4](https://ull.tju.edu.cn/education/manual/verdi.manual_v5.pdf#page=34).

Use these serial checks alongside the front panel:

| Check | Query | Expected before key-on |
| --- | --- | --- |
| Laser state | `?L` | `0` — STANDBY |
| Keyswitch | `?K` | `0` — OFF/STANDBY |
| Shutter | `?S` | `0` — CLOSED during preparation |
| Active faults | `?F` | No active fault; confirm front panel and verified reply meaning |
| LBO temperature servo | `?LBOSS` | `1` — LOCKED |
| LBO temperature | `?LBOT`, `?LBOST` | Measured temperature settled at the target |
| Diode temperature servo | `?D1SS` | `1` — LOCKED |
| Etalon temperature servo | `?ESS` | `1` — LOCKED |
| Vanadate temperature servo | `?VSS` | `1` — LOCKED |
| Requested output | `?SP` | Confirm the intended, approved power |

Servo codes: [manual, Table 5-4](https://ull.tju.edu.cn/education/manual/verdi.manual_v5.pdf#page=55).
The previously reported `?LBOSS = 2` (SEEKING) and LBO measured/setpoint values
of **118.25 / 149.20 °C** were not ready for key-on. Those are an earlier example,
not live readings; use the fresh replies below. No temperature tolerance is guessed.

### Verify physically

- Cooling operating with no leaks; approximately **20 °C** for water-cooled configurations.
- Output blocked or directed at the intended target.
- Everyone present wearing suitable laser safety eyewear.
- Interlocks functional and ventilation unobstructed.

[Manual, §§3–4](https://ull.tju.edu.cn/education/manual/verdi.manual_v5.pdf).

The next cell refreshes the serial checks once. It preserves raw bytes and makes
no automatic readiness decision. No key, shutter or laser-enable action is sent.
If warm-up is incomplete, leave the key in STANDBY. After a successful exchange,
you may deliberately rerun this read-only cell later to inspect progress; do not
rerun after a communication failure. Confirm front-panel and physical readiness
with the operator before any key-on action.

In [5]:
readiness_queries = {
    b"?L": "Laser state",
    b"?K": "Keyswitch",
    b"?S": "Shutter",
    b"?F": "Active faults",
    b"?LBOSS": "LBO temperature servo",
    b"?LBOT": "LBO measured temperature",
    b"?LBOST": "LBO temperature setpoint",
    b"?D1SS": "Diode temperature servo",
    b"?ESS": "Etalon temperature servo",
    b"?VSS": "Vanadate temperature servo",
    b"?SP": "Requested output",
}

pre_key_on = {}
for query, label in readiness_queries.items():
    reply = laser.read(query)
    pre_key_on[query] = reply
    print(f"{label:30s}: {reply!r}")

Laser state                   : b'0'
Keyswitch                     : b'0'
Shutter                       : b'0'
Active faults                 : b'SYSTEM OK'
LBO temperature servo         : b'1'
LBO measured temperature      : b'149.16'
LBO temperature setpoint      : b'149.20'
Diode temperature servo       : b'1'
Etalon temperature servo      : b'1'
Vanadate temperature servo    : b'1'
Requested output              : b'0.0100'


## 6. Inspect an error

Print the last transaction without sending anything, then disconnect.
An incomplete exchange stops the session; reconnecting alone does not prove alignment.

In [6]:
print(laser.last_transaction)

{'instruction': b'?SP', 'request': b'?SP\r\n', 'response': b'0.0100\r\nVerdi V-2>', 'payload': b'0.0100', 'outcome': 'received', 'error': None}


## 7. Disconnect

Release the port without changing laser state. No readiness verdict or key-on
action has been issued. Complete the front-panel and physical checks with the
operator before any key-on action. Continue to [lesson 5](05_set_power.ipynb)
only under the approved operating procedure.

In [7]:
laser.disconnect()